# Punto 2
## importación de librerías

In [1]:
import pandas as pd
from sqlalchemy import create_engine

## importación del CSV a SQLite

In [2]:
# Cargar dataset
df = pd.read_csv(
    "../data/ventas_techventas.csv",
    encoding="latin1"
)

# Crear conexión SQLite
engine = create_engine(
    "sqlite:///../output/techventas.db"
)

# Exportar tabla ventas
df.to_sql(
    "ventas",
    engine,
    if_exists="replace",
    index=False
)

print("Tabla ventas creada correctamente.")

Tabla ventas creada correctamente.


## Creación de la tabla vendedores

In [3]:
vendedores = pd.DataFrame({
    "vendedor": [
        "Ana López",
        "Carlos Ruiz",
        "Luis Mora",
        "Maria Torres"
    ],
    "zona": [
        "Norte",
        "Sur",
        "Norte",
        "Centro"
    ],
    "meta_mensual": [
        8000,
        7500,
        8000,
        7000
    ]
})

vendedores.to_sql(
    "vendedores",
    engine,
    if_exists="replace",
    index=False
)

vendedores

,vendedor,zona,meta_mensual
0,Ana López,Norte,8000
1,Carlos Ruiz,Sur,7500
2,Luis Mora,Norte,8000
3,Maria Torres,Centro,7000


## Ejecución en el notebook con pd.read_sql()
## Q1

In [4]:
query = """
SELECT DISTINCT producto AS producto_unico
FROM ventas;
"""

pd.read_sql(query, engine)

,producto_unico
0,Laptop Pro 15
1,Mouse Inalambrico
2,NaN
3,Teclado Mecanico
4,SSD 1TB
5,Router WiFi 6


## Q2

In [5]:
query = """
SELECT *
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
AND cantidad > 5;
"""

pd.read_sql(query, engine)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
1,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
2,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres
3,1008,2024-01-22,C006,NetPrime,Norte,Router WiFi 6,Redes,8.0,120.0,0.00,Luis Mora
4,1009,2024-01-25,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12.0,95.0,0.10,Maria Torres
5,1012,2024-02-05,C008,BetaSoft,Centro,Mouse Inalambrico,Perifericos,30.0,25.0,0.05,Maria Torres
6,1015,2024-02-12,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,25.0,95.0,0.10,Maria Torres
7,1018,2024-02-20,C002,Innovatek,Sur,SSD 1TB,Almacenamiento,18.0,95.0,0.00,Carlos Ruiz
8,1019,2024-02-22,C007,AlphaTech,Sur,Mouse Inalambrico,Perifericos,12.0,25.0,0.00,Maria Torres
9,1022,2024-03-04,C001,TechCorp SA,Norte,SSD 1TB,Almacenamiento,30.0,95.0,0.15,Carlos Ruiz


## Q3

In [6]:
query = """
SELECT
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto_total
FROM ventas
GROUP BY vendedor
HAVING ingreso_bruto_total > 10000;
"""

pd.read_sql(query, engine)

,vendedor,ingreso_bruto_total
0,Ana Lï¿½ï¿,20810.0
1,Carlos Ruiz,21355.0
2,Luis Mora,19830.0
3,Maria Torres,11860.0


## Q4

In [7]:
query = """
SELECT
    categoria,
    COUNT(*) AS total_pedidos,
    SUM(cantidad) AS unidades_vendidas,
    AVG(precio_unitario) AS precio_promedio
FROM ventas
GROUP BY categoria;
"""

pd.read_sql(query, engine)

,categoria,total_pedidos,unidades_vendidas,precio_promedio
0,NaN,11,NaN,NaN
1,Almacenamiento,12,260.0,95.000000
2,Electronica,14,31.0,1139.285714
3,Perifericos,15,215.0,53.000000
4,Redes,8,39.0,120.000000


## Q5

In [8]:
query = """
SELECT
    v.vendedor,
    v.zona,
    v.meta_mensual,
    SUM(
        ve.cantidad * ve.precio_unitario
    ) AS ventas_totales,
    ROUND(
        (
            SUM(
                ve.cantidad * ve.precio_unitario
            ) * 100.0
        ) / v.meta_mensual,
        2
    ) AS porcentaje_cumplimiento
FROM vendedores v
JOIN ventas ve
    ON v.vendedor = ve.vendedor
GROUP BY
    v.vendedor,
    v.zona,
    v.meta_mensual;
"""

pd.read_sql(query, engine)

,vendedor,zona,meta_mensual,ventas_totales,porcentaje_cumplimiento
0,Carlos Ruiz,Sur,7500,21355.0,284.73
1,Luis Mora,Norte,8000,19830.0,247.88
2,Maria Torres,Centro,7000,11860.0,169.43


## Q6

In [9]:
query = """
SELECT
    cliente_id,
    ingreso_total
FROM (
    SELECT
        cliente_id,
        SUM(
            cantidad * precio_unitario
        ) AS ingreso_total
    FROM ventas
    WHERE fecha BETWEEN
          '2024-01-01'
      AND '2024-06-30'
    GROUP BY cliente_id
)
ORDER BY ingreso_total DESC
LIMIT 1;
"""

pd.read_sql(query, engine)

,cliente_id,ingreso_total
0,C001,19025.0
